# Phase 2: Financial KPI Dashboard

**Days 13-15**: KPI Calculations and Visualizations

This notebook calculates and displays all 5 financial KPIs for each user.

## CELL 1: Setup & Imports

**⚠️ REPLACE this cell completely:**

In [ ]:
import pandas as pd
import numpy as np
import sys
from sqlalchemy import create_engine

sys.path.insert(0, '../src')

from pfm.config import DB_PATH, USERS, NEEDS_CATEGORIES, WANTS_CATEGORIES, SAVINGS_CATEGORIES
from pfm.analytics.kpi_engine import (
    savings_rate,
    debt_to_income_ratio,
    budget_variance,
    emergency_fund_coverage,
    spending_50_30_20
)

engine = create_engine(f'sqlite:///{DB_PATH}')

print('✅ KPI Dashboard Ready')

## CELL 2: Load Data

**Copy as-is:**

In [ ]:
df = pd.read_sql('SELECT * FROM transactions', engine)
df['date'] = pd.to_datetime(df['date'])
budgets = pd.read_sql('SELECT * FROM budgets', engine)

print(f'Data loaded: {len(df):,} transactions, {df["user_name"].nunique()} users')

## CELL 3: KPI Dashboard - All Users

**Copy as-is:**

In [ ]:
print('\n' + '='*80)
print('💰 FINANCIAL KPI DASHBOARD - ALL USERS')
print('='*80)

kpi_results = []

for user_info in USERS:
    user_id = user_info['user_id']
    user_name = user_info['user_name']
    
    print(f'\n\n👤 {user_name}')
    print('-'*80)
    
    # KPI 1: Savings Rate
    sr = savings_rate(df, user_id)
    print(f'\n1️⃣ SAVINGS RATE')
    print(f'   Rate: {sr["rate"]:.1f}%')
    print(f'   Income: Rs. {sr["income"]:,.0f}')
    print(f'   Expenses: Rs. {sr["expenses"]:,.0f}')
    print(f'   Savings: Rs. {sr["savings"]:,.0f}')
    status = '✅ Good' if sr['rate'] >= 20 else '⚠️ Below target'
    print(f'   Status: {status} (target: 20%+)')
    
    # KPI 2: Debt-to-Income Ratio
    dti = debt_to_income_ratio(df, user_id)
    print(f'\n2️⃣ DEBT-TO-INCOME RATIO')
    print(f'   DTI: {dti["dti"]:.1f}%')
    print(f'   Monthly Income: Rs. {dti["monthly_income"]:,.0f}')
    print(f'   Monthly Debt: Rs. {dti["monthly_debt"]:,.0f}')
    status = '✅ Healthy' if dti['dti'] <= 36 else '⚠️ High'
    print(f'   Status: {status} (target: <36%)')
    
    # KPI 3: Emergency Fund Coverage
    ef = emergency_fund_coverage(df, user_id, months=3)
    print(f'\n3️⃣ EMERGENCY FUND COVERAGE')
    print(f'   Coverage: {ef["coverage_months"]:.1f}x months')
    print(f'   Liquid Savings: Rs. {ef["liquid_savings"]:,.0f}')
    print(f'   Avg Monthly Expense: Rs. {ef["avg_monthly_expense"]:,.0f}')
    print(f'   Target (3 months): Rs. {ef["target_emergency_fund"]:,.0f}')
    status = '✅ Adequate' if ef['coverage_months'] >= 1 else '⚠️ Below target'
    print(f'   Status: {status}')
    
    # KPI 4: 50/30/20 Rule
    rule = spending_50_30_20(df, user_id, NEEDS_CATEGORIES, WANTS_CATEGORIES, SAVINGS_CATEGORIES)
    print(f'\n4️⃣ 50/30/20 RULE')
    print(f'   Needs: {rule["needs_pct"]:.0f}% (target: 50%)')
    print(f'   Wants: {rule["wants_pct"]:.0f}% (target: 30%)')
    print(f'   Savings: {rule["savings_pct"]:.0f}% (target: 20%)')
    needs_ok = rule['needs_pct'] <= 50
    wants_ok = rule['wants_pct'] <= 30
    print(f'   Needs Status: {"✅" if needs_ok else "⚠️"}')
    print(f'   Wants Status: {"✅" if wants_ok else "⚠️"}')
    
    # KPI 5: Budget Variance
    bv = budget_variance(df, budgets, user_id)
    if len(bv) > 0:
        avg_variance_pct = bv['variance_pct'].mean()
        overbudget_count = (bv['variance'] > 0).sum()
        print(f'\n5️⃣ BUDGET VARIANCE')
        print(f'   Avg Variance: {avg_variance_pct:+.1f}%')
        print(f'   Categories Over Budget: {overbudget_count}')
        print(f'   Total Variance: Rs. {bv["variance"].sum():+,.0f}')
        status = '✅ On track' if avg_variance_pct <= 10 else '⚠️ Over budget'
        print(f'   Status: {status}')
    
    kpi_results.append({
        'user': user_name,
        'savings_rate': sr['rate'],
        'dti': dti['dti'],
        'emergency_fund': ef['coverage_months'],
        'needs_pct': rule['needs_pct'],
        'wants_pct': rule['wants_pct']
    })

print('\n' + '='*80)
print('✅ KPI Dashboard Complete')
print('='*80)

## CELL 4: KPI Summary Table

**Copy as-is:**

In [ ]:
kpi_df = pd.DataFrame(kpi_results)

print('\n📊 KPI SUMMARY TABLE')
print(kpi_df.to_string(index=False))

print('\n\n📈 COMPARATIVE ANALYSIS')
print(f'\nSavings Rate:')
print(f'  Highest: {kpi_df.loc[kpi_df["savings_rate"].idxmax(), "user"]} ({kpi_df["savings_rate"].max():.1f}%)')
print(f'  Lowest: {kpi_df.loc[kpi_df["savings_rate"].idxmin(), "user"]} ({kpi_df["savings_rate"].min():.1f}%)')

print(f'\nDebt-to-Income Ratio:')
print(f'  Highest: {kpi_df.loc[kpi_df["dti"].idxmax(), "user"]} ({kpi_df["dti"].max():.1f}%)')
print(f'  Lowest: {kpi_df.loc[kpi_df["dti"].idxmin(), "user"]} ({kpi_df["dti"].min():.1f}%)')

print(f'\nEmergency Fund Coverage:')
print(f'  Highest: {kpi_df.loc[kpi_df["emergency_fund"].idxmax(), "user"]} ({kpi_df["emergency_fund"].max():.1f}x months)')
print(f'  Lowest: {kpi_df.loc[kpi_df["emergency_fund"].idxmin(), "user"]} ({kpi_df["emergency_fund"].min():.1f}x months)')

## CELL 5: Key Insights

**Copy as-is:**

In [ ]:
print('\n' + '='*80)
print('💡 KEY INSIGHTS & RECOMMENDATIONS')
print('='*80)

# Best performer
best_saver = kpi_df.loc[kpi_df['savings_rate'].idxmax()]
print(f'\n🏆 Best Saver: {best_saver["user"]} ({best_saver["savings_rate"]:.1f}% savings rate)')
print('   Recommendation: Use as model for financial discipline')

# Needs attention
highest_dti = kpi_df.loc[kpi_df['dti'].idxmax()]
if highest_dti['dti'] > 36:
    print(f'\n⚠️  High Debt Concern: {highest_dti["user"]} (DTI: {highest_dti["dti"]:.1f}%)')
    print('   Recommendation: Focus on debt repayment strategy')

# Emergency fund
low_ef = kpi_df[kpi_df['emergency_fund'] < 1]
if len(low_ef) > 0:
    print(f'\n🆘 Emergency Fund Risk: {len(low_ef)} user(s) below 1 month coverage')
    print('   Recommendation: Build emergency fund as priority')
else:
    print(f'\n✅ Emergency Fund: All users have adequate coverage')

# 50/30/20 rule
over_wants = kpi_df[kpi_df['wants_pct'] > 30]
if len(over_wants) > 0:
    print(f'\n📊 Wants Overspending: {len(over_wants)} user(s) exceed 30% wants budget')
    for _, user in over_wants.iterrows():
        print(f'   - {user["user"]}: {user["wants_pct"]:.0f}% wants spending')
    print('   Recommendation: Review discretionary spending categories')

print('\n' + '='*80)
print('✅ Phase 2 Days 13-15 Complete: KPI Dashboard Generated')
print('='*80)